# NHANES Glucose Prediction Pipeline

This notebook runs the full pipeline: **upload → audit → pre-processing → (optional) statistical significance filtering → pre-processing (scaler + target trim) → split → train NN → evaluate**.

Run all cells in order. Change the config below to enable optional steps (e.g. `use_statistical_significance_filter: true` for p < 0.1, or `use_hyperparameter_optimization: "optuna"`).

In [10]:
import os
import sys
import yaml
import pandas as pd
import numpy as np

# Use the current notebook working directory directly.
NB_DIR = os.path.abspath(".")
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)

# Config (mirrors config.yaml; override here or load from file)
CONFIG = {
    "data_path": "nhanes_1999_2018_yayhoo_fasting_diet_imputed.csv",
    "target_col": "glucose",
    "feature_cols": [],
    "use_statistical_significance_filter": False,
    "feature_selection_p_value_threshold": 0.1,
    "train_size": 0.7,
    "val_size": 0.15,
    "test_size": 0.15,
    "random_state": 42,
    "target_trim_lower_percentile": 0.05,
    "target_trim_upper_percentile": 0.95,
    "imputation": "median",
    "nn": {
        "epochs": 200,
        "batch_size": 256,
        "lr": 0.0015,
        "weight_decay": 0.0002,
        "patience": 30,
        "dropout": 0.1,
        "num_layers": 2,
        "layer_width": 32,
        "architecture_pattern": "constant",
        "activation": "relu",
    },
    "use_hyperparameter_optimization": "none",
    "optuna": {"n_trials": 30, "timeout": 600},
    "output_dir": "outputs",
}

# Optional: load from file
# with open(os.path.join(NB_DIR, "config.yaml")) as f:
#     CONFIG = yaml.safe_load(f)
CONFIG

{'data_path': 'nhanes_1999_2018_yayhoo_fasting_diet_imputed.csv',
 'target_col': 'glucose',
 'feature_cols': [],
 'use_statistical_significance_filter': False,
 'feature_selection_p_value_threshold': 0.1,
 'train_size': 0.7,
 'val_size': 0.15,
 'test_size': 0.15,
 'random_state': 42,
 'target_trim_lower_percentile': 0.05,
 'target_trim_upper_percentile': 0.95,
 'imputation': 'median',
 'nn': {'epochs': 200,
  'batch_size': 256,
  'lr': 0.0015,
  'weight_decay': 0.0002,
  'patience': 30,
  'dropout': 0.1,
  'num_layers': 2,
  'layer_width': 32,
  'architecture_pattern': 'constant',
  'activation': 'relu'},
 'use_hyperparameter_optimization': 'none',
 'optuna': {'n_trials': 30, 'timeout': 600},
 'output_dir': 'outputs'}

## Step 1: Load data

In [11]:
from steps.upload import load_csv

data_path = CONFIG["data_path"]
if not os.path.isabs(data_path):
    data_path = os.path.join(NB_DIR, data_path)
df = load_csv(data_path)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

Loaded 21849 rows, 29 columns


,SEQN,cycle_begin_year,age,gender,bp_sys,bp_di,weight,height,bmi,waist,...,glucose,triglycerides,meds_hbp,meds_chol,imputed_weight,imputed_height,imputed_bmi,imputed_waist,imputed_bp_sys,imputed_bp_di
0,9966.0,2001,39.0,male,128.0,8.000000e+01,91.7,174.2,30.22,100.8,...,102.4,417.0,NaN,NaN,False,False,False,False,False,False
1,9968.0,2001,84.0,female,120.0,5.397605e-79,51.7,144.9,24.62,86.1,...,96.4,130.0,True,NaN,False,False,False,False,False,False
2,9969.0,2001,51.0,female,120.0,7.200000e+01,58.0,161.4,22.26,74.0,...,90.7,92.0,NaN,NaN,False,False,False,False,False,False
3,9972.0,2001,44.0,male,146.0,8.600000e+01,101.5,173.2,33.84,116.0,...,96.4,170.0,False,NaN,False,False,False,False,False,False
4,9973.0,2001,63.0,female,100.0,7.400000e+01,64.9,156.2,26.60,94.2,...,83.7,244.0,NaN,NaN,False,False,False,False,False,False


## Step 2: Audit

In [12]:
from steps.audit import run_audit

audit = run_audit(df)
print("Missing:", len(audit["missing"]))
print("Duplicates:", audit["duplicates"])
print("Constant cols:", audit["constant_cols"])
print("ID-like cols:", audit["id_like_cols"])

Missing: 2
Duplicates: 0
Constant cols: []
ID-like cols: []


## Step 3: First pre-processing

In [13]:
from steps.preprocess1 import preprocess1

target_col = CONFIG["target_col"]
df = preprocess1(df, target_col, drop_duplicates=True, drop_constant_cols=False)
print(f"Rows after clean: {len(df)}")

Rows after clean: 21849


## Step 4: Feature selection (optional)

When `use_statistical_significance_filter` is True, we keep only features with **p < 0.1** (may dramatically reduce the number of features).

In [14]:
from steps.audit import get_numeric_columns
from steps.feature_selection import select_features_by_pvalue

use_filter = CONFIG.get("use_statistical_significance_filter", False)
p_threshold = CONFIG.get("feature_selection_p_value_threshold", 0.1)
feature_cols = CONFIG.get("feature_cols") or []

if use_filter:
    numeric = get_numeric_columns(df)
    candidates = [c for c in numeric if c != target_col]
    feature_cols = select_features_by_pvalue(df, target_col, candidate_features=candidates, p_threshold=p_threshold)
    print(f"Statistical filter (p < {p_threshold}): selected {len(feature_cols)} features")
else:
    if not feature_cols:
        numeric = get_numeric_columns(df)
        feature_cols = [c for c in numeric if c != target_col]
    print(f"Using {len(feature_cols)} features (no filter)")
feature_cols

Using 19 features (no filter)


['SEQN',
 'cycle_begin_year',
 'age',
 'bp_sys',
 'bp_di',
 'weight',
 'height',
 'bmi',
 'waist',
 'kcal',
 'protein',
 'sugar',
 'carb',
 'fat_total',
 'fat_sat',
 'fat_mon',
 'fat_poly',
 'hdl',
 'triglycerides']

## Step 5: Second pre-processing (scaler, target trim) and split

In [15]:
from steps.preprocess2 import preprocess2_split

scaler, imputer, X_train, X_val, X_test, y_train, y_val, y_test, _ = preprocess2_split(
    df, target_col, feature_cols,
    train_size=CONFIG["train_size"], val_size=CONFIG["val_size"], test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    target_trim_lower=CONFIG["target_trim_lower_percentile"],
    target_trim_upper=CONFIG["target_trim_upper_percentile"],
    imputation=CONFIG["imputation"],
)
print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

Train: 13848, Val: 2968, Test: 2968


## Step 6: Train neural network

Optional: set `use_hyperparameter_optimization: "optuna"` in config and re-run the config cell to run Optuna search first.

In [16]:
from steps.train import train_nn, _hidden_layers_from_config
from models.nn_whuber import NNWeightedHuberWrapper

nn_cfg = CONFIG.get("nn", {})
use_hpo = CONFIG.get("use_hyperparameter_optimization", "none")
random_state = CONFIG["random_state"]

if use_hpo == "optuna":
    try:
        import optuna
        optuna_cfg = CONFIG.get("optuna", {}) or {}
        n_trials = optuna_cfg.get("n_trials", 30)
        timeout = optuna_cfg.get("timeout", 600)
        def objective(trial):
            num_layers = trial.suggest_int("num_layers", 1, 4)
            layer_width = trial.suggest_int("layer_width", 16, 128)
            pattern = trial.suggest_categorical("architecture_pattern", ["constant", "pyramid", "funnel"])
            activation = trial.suggest_categorical("activation", ["relu", "tanh", "elu"])
            dropout = trial.suggest_float("dropout", 0.0, 0.4)
            epochs = trial.suggest_int("epochs", 80, 250)
            batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
            lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
            weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
            patience = trial.suggest_int("patience", 15, 40)
            hidden = _hidden_layers_from_config(num_layers, layer_width, pattern)
            m = NNWeightedHuberWrapper(hidden_layers=hidden, dropout=dropout, task_type="regression", activation=activation)
            res = m.fit(X_train, y_train, X_val, y_val, epochs=epochs, batch_size=batch_size, lr=lr,
                        weight_decay=weight_decay, patience=patience, random_seed=random_state)
            val_rmse = res.get("history", {}).get("val_rmse", [])
            return val_rmse[-1] if val_rmse else float("inf")
        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=n_trials, timeout=timeout, show_progress_bar=True)
        best = study.best_params
        nn_cfg = {**nn_cfg, **best}
        print(f"Best val RMSE: {study.best_value:.4f}")
    except ImportError:
        print("Optuna not installed; using config defaults.")

model, results = train_nn(X_train, y_train, X_val, y_val, nn_cfg, random_state=random_state)
y_pred = model.predict(X_test)
print("Training complete.")

Training complete.


## Step 7: Performance assessment

In [17]:
from steps.evaluate import (
    evaluate_and_save,
    save_learning_curve,
    save_pred_vs_actual,
    save_residual_plot,
    save_results_table,
)

output_dir = CONFIG.get("output_dir", "outputs")
if not os.path.isabs(output_dir):
    output_dir = os.path.join(NB_DIR, output_dir)
metrics = evaluate_and_save(y_test, y_pred, output_dir, "metrics_notebook.json",
                            extra={"feature_cols": feature_cols})
history = getattr(model, "history", None) or results.get("history", {})
if history:
    save_learning_curve(history, output_dir)
save_pred_vs_actual(y_test, y_pred, output_dir)
save_residual_plot(y_test, y_pred, output_dir)
save_results_table(metrics, output_dir, extra={"n_train": len(y_train), "n_test": len(y_test), "n_features": len(feature_cols)})
print("Test set metrics:", metrics)
print("Plots and results_table saved to", output_dir)

Test set metrics: {'MAE': 8.861915905584866, 'RMSE': 12.407970520796498, 'R2': 0.23517670999315443, 'MedianAE': 6.460670471191406}
Plots and results_table saved to c:\Users\nolan.hedglin\Downloads\nhanes_glucose_pipeline\outputs


## Optional: Predictions vs actual (simple plot)

In [18]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.3, s=5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", label="y=x")
plt.xlabel("Actual glucose")
plt.ylabel("Predicted glucose")
plt.title("Predictions vs actual (test set)")
plt.legend()
plt.tight_layout()
plt.show()

C:\Users\nolan.hedglin\AppData\Local\Temp\ipykernel_22988\2266463681.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
